In [28]:
import re
import pandas as pd
from docx import Document
from pathlib import Path

#### 1. Defining file paths
Specify the paths to the five CISA manuals and check that all files exist.

In [14]:
data_dir = Path(r"C:\Users\liisiparve\Projects\cybersecurity-project\data\raw")

files = {
    "Healthcare": ("healthcare_and_public_health.docx", "HC"),
    "Vendor": ("vendor_supply_chain_compromise.docx", "VD"),
    "Ransomware": ("ransomware.docx", "RW"),
    "Insider Threat": ("insider_threat.docx", "IT"),
    "Executives": ("executives_and_senior_leadership.docx", "EX")
}

for name, (filename, prefix) in files.items():
    path = data_dir / filename
    print(f"{name}: {path.exists()}")

Healthcare: True
Vendor: True
Ransomware: True
Insider Threat: True
Executives: True


#### 2. Loading the documents
Load all five Word documents and display their paragraph counts.

In [18]:
documents = {}

for name, (filename, prefix) in files.items():
    documents[name] = Document(data_dir / filename)
    print(f"{name}: {len(documents[name].paragraphs)} paragraphs")

Healthcare: 331 paragraphs
Vendor: 304 paragraphs
Ransomware: 330 paragraphs
Insider Threat: 293 paragraphs
Executives: 209 paragraphs


#### 3. Inspecting document structure
Find module, vignette, and day headings to understand how the scenarios are organized.

In [29]:
for name, doc in documents.items():
    print(f"\n--- {name} ---")

    for i, p in enumerate(doc.paragraphs):
        text = p.text.strip()

        if re.fullmatch(
            r"(Module [12].*|Vignette [12]:.*|Day \d+.*)",
            text
        ):
            print(i, repr(text))


--- Healthcare ---
16 'Module 1\t8'
17 'Module 2\t11'
72 'Module 1: This module introduces several events, including a suspicious email to employees and an unannounced vendor.'
73 'Module 2: This module includes a ransomware attack and the discovery of patient data theft.'
86 'Module 1'
87 'Day 1'
89 'Day 9'
117 'Day 10'
119 'Day 13'
126 'Day 15'
134 'Module 2'
135 'Day 20'
137 'Day 23 - Morning'
139 'Day 23 - Afternoon'
156 'Day 25'
159 'Day 26'
163 'Day 27'

--- Vendor ---
15 'Module 1\t9'
16 'Module 2\t11'
72 'Module 1: This module introduces a state of the world describing a global cybersecurity incident impacting Managed Service Providers (MSPs) and your organization acquiring a new MSP.'
73 'Module 2: This module covers the downstream impacts of a cyber incident at your MSP.'
110 'Module 1'
111 'Day 1'
122 'Day 7'
136 'Day 8'
143 'Day 10'
149 'Module 2 – 6 months later'
150 'Day 182 - Morning'
152 'Day 182 - Afternoon'
162 'Day 182 – Evening'
167 'Day 185'

--- Ransomware ---
12

#### 4. Extracting scenario events and discussion questions
Define a function that identifies scenario events and discussion questions in each CISA manual and organizes them into two DataFrames.

In [30]:
def parse_manual(path, manual_name, prefix):
    doc = Document(path)

    # keep only the paragraphs that aren't empty
    paragraphs = [
        (i, p.text.strip())
        for i, p in enumerate(doc.paragraphs)
        if p.text.strip()
    ]

    events = []
    questions = []

    section = None # current module or vignette
    day = None # current scenario day
    mode = None # whether we're reading events or questions
    event_parts = [] # paragraphs belonging to the current event
    event_index = None # paragraph index where the event starts
    started = False # whether we've reached the actual exercise

    def save_event():
        if event_parts and section is not None:
            events.append({
                "manual_title": manual_name,
                "inject_id": f"{prefix}-INJ-{event_index}",
                "section": section,
                "day": day,
                "event_description": " ".join(event_parts),
                "source_paragraph": event_index
            })

    for idx, text in paragraphs:

        # find the beginning of the actual exercise
        if not started:
            if manual_name == "Executives":
                start_heading = "General Discussion Questions"
            else:
                start_heading = "Module 1"

            if text == start_heading:
                started = True
                section = text

                if manual_name == "Executives":
                    mode = "question"
                else:
                    mode = "event"

            continue

        # stop before the appendices
        if re.fullmatch(
            r"Appendix\s+A\s*:.*",
            text,
            re.I
        ):
            save_event()
            break

        # recognize modules and vignettes
        is_module = re.fullmatch(
            r"Module\s+[12](?:\s*[–-].*)?",
            text,
            re.I
        )

        is_vignette = re.fullmatch(
            r"Vignette\s+[12]\s*:.*",
            text,
            re.I
        )

        if is_module or is_vignette:
            save_event()

            section = text
            day = None
            mode = "event"
            event_parts = []
            event_index = None
            continue

        # recognize discussion question headings
        if text == "Discussion Questions":
            save_event()
            event_parts = []
            mode = "question"
            continue

        # recognize scenario days
        if re.fullmatch(r"Day\s+\d+.*", text, re.I):
            save_event()

            day = text
            event_index = idx
            event_parts = []
            mode = "event"
            continue

        # recognize scenario headings in Executives manual
        if text == "Scenario" and manual_name == "Executives":
            mode = "event"
            continue

        # skip introductory descriptions of questions
        if (
            text.lower().startswith(
                "discussion questions included"
            )
            or text.lower().startswith(
                "the general discussion questions below"
            )
        ):
            continue

        # recognize questions and discussion prompts
        is_prompt = (
            "?" in text
            or bool(re.match(
                r"^(Describe|Discuss|Explain|Identify|List|"
                r"Based on|Using your|What|How|Who|When|"
                r"Where|Does|Do|Are|Is|Have|Has|"
                r"Can|Could|Would|Should)\b",
                text,
                re.I
            ))
        )

        # if a question follows an event, switch modes
        if mode == "event" and is_prompt:
            save_event()
            event_parts = []
            mode = "question"

        # save discussion questions
        if mode == "question":
            if is_prompt:
                questions.append({
                    "manual_title": manual_name,
                    "question_id": f"{prefix}-Q-{idx}",
                    "section": section,
                    "day": day,
                    "question_text": text,
                    "domain_category": None,
                    "difficulty_level": None,
                    "required_skill": None,
                    "source_paragraph": idx
                })

        # save event descriptions
        elif mode == "event":
            if manual_name != "Executives" and day is None:
                continue

            if event_index is None:
                event_index = idx

            event_parts.append(text)

    return pd.DataFrame(events), pd.DataFrame(questions)

#### 5. Extracting data from all five manuals
Apply the extraction function to each CISA manual, store the resulting DataFrames, and display the number of events and questions extracted.

In [22]:
all_events = []
all_questions = []

for name, (filename, prefix) in files.items():
    path = data_dir / filename

    events, questions = parse_manual(
        path,
        name,
        prefix
    )

    all_events.append(events)
    all_questions.append(questions)

    print(f"\n{name}")
    print("Events:", len(events))
    print("Questions:", len(questions))


Healthcare
Events: 11
Questions: 61

Vendor
Events: 8
Questions: 49

Ransomware
Events: 8
Questions: 58

Insider Threat
Events: 12
Questions: 52

Executives
Events: 2
Questions: 48


#### 6. Inspecting extracted data
Display the first few events and discussion questions from each manual to check whether the extraction worked correctly.

In [33]:
for name, events, questions in zip(
    files.keys(),
    all_events,
    all_questions
):
    print(f"\n--- {name} ---")

    print("\nEvents:")
    display(events.head(3))

    print("\nDiscussion questions:")
    display(questions.head(3))


--- Healthcare ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Healthcare,HC-INJ-87,Module 1,Day 1,The Cybersecurity and Infrastructure Security ...,87
1,Healthcare,HC-INJ-89,Module 1,Day 9,<Organization> employees receive e-mail notifi...,89
2,Healthcare,HC-INJ-117,Module 1,Day 10,Manufacturer’s updates are pushed for installa...,117



Discussion questions:


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Healthcare,HC-Q-93,Module 1,Day 9,What are the greatest cyber threats to your or...,None,None,None,93
1,Healthcare,HC-Q-94,Module 1,Day 9,What cybersecurity threat information does you...,None,None,None,94
2,Healthcare,HC-Q-95,Module 1,Day 9,What cyber threat information is most useful?,None,None,None,95



--- Vendor ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Vendor,VD-INJ-111,Module 1,Day 1,CISA and the Federal Bureau of Investigation (...,111
1,Vendor,VD-INJ-122,Module 1,Day 7,Your organization contracts a new MSP who is k...,122
2,Vendor,VD-INJ-136,Module 1,Day 8,Data from multiple sector partners is found on...,136


... (10 outputs left out)


--- Executives ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Executives,EX-INJ-109,Vignette 1: Whaling/Ransomware,None,A senior finance employee receives what appear...,109
1,Executives,EX-INJ-126,Vignette 2: Insider Threat,None,Your IT department determines that a known exp...,126



Discussion questions:


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Executives,EX-Q-86,General Discussion Questions,None,What are the greatest cyber threats to your or...,None,None,None,86
1,Executives,EX-Q-87,General Discussion Questions,None,Describe your cybersecurity culture.,None,None,None,87
2,Executives,EX-Q-88,General Discussion Questions,None,How do you articulate the importance of a stro...,None,None,None,88


#### 7. Combining extracted data
Combine the events and discussion questions from all five manuals into two DataFrames and display their dimensions and first few rows.

In [35]:
df_injects = pd.concat(
    all_events,
    ignore_index=True
)

df_questions = pd.concat(
    all_questions,
    ignore_index=True
)

print("Combined events:", df_injects.shape)
print("Combined questions:", df_questions.shape)

display(df_injects.head())
display(df_questions.head())

Combined events: (41, 6)
Combined questions: (268, 9)


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Healthcare,HC-INJ-87,Module 1,Day 1,The Cybersecurity and Infrastructure Security ...,87
1,Healthcare,HC-INJ-89,Module 1,Day 9,<Organization> employees receive e-mail notifi...,89
2,Healthcare,HC-INJ-117,Module 1,Day 10,Manufacturer’s updates are pushed for installa...,117
3,Healthcare,HC-INJ-119,Module 1,Day 13,A third-party electronic medical record (EMR) ...,119
4,Healthcare,HC-INJ-126,Module 1,Day 15,Radiology reports an issue with their Magnetic...,126


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Healthcare,HC-Q-93,Module 1,Day 9,What are the greatest cyber threats to your or...,None,None,None,93
1,Healthcare,HC-Q-94,Module 1,Day 9,What cybersecurity threat information does you...,None,None,None,94
2,Healthcare,HC-Q-95,Module 1,Day 9,What cyber threat information is most useful?,None,None,None,95
3,Healthcare,HC-Q-96,Module 1,Day 9,How is information disseminated throughout you...,None,None,None,96
4,Healthcare,HC-Q-97,Module 1,Day 9,What actions would your organization take in r...,None,None,None,97


#### 8. Exporting the datasets
Save the combined events and discussion questions as CSV files in the processed data folder.

In [37]:
output_dir = data_dir.parent / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

df_injects.to_csv(
    output_dir / "cisa_scenario_injects.csv",
    index=False,
    encoding="utf-8-sig"
)

df_questions.to_csv(
    output_dir / "cisa_discussion_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV files saved successfully!")
print("Total events:", len(df_injects))
print("Total questions:", len(df_questions))

CSV files saved successfully!
Total events: 41
Total questions: 268


#### 9. Validating extracted data
Check that the extracted questions match the original Word documents and that there are no duplicate IDs.

In [39]:
# verify that each question matches its original paragraph
for index, row in df_questions.iterrows():
    doc = documents[row["manual_title"]]
    original = doc.paragraphs[int(row["source_paragraph"])].text.strip()

    assert original == row["question_text"], f"Mismatch: {row['question_id']}"

# check that all record IDs are unique
assert df_injects["inject_id"].is_unique
assert df_questions["question_id"].is_unique

print("All validation checks passed!")

All validation checks passed!
